# DataObject — Lazy, Alias-Driven Structured Access to Sensor Data

The `DataObject` class bridges query metadata to time-series data with predictable, alias-based column names and built-in entity grouping.

**Lazy by default**: calling `query.data()` only fetches metadata and per-series stats (row count, time range). The actual time-series data is fetched on demand when you call `.dataframe()`, `data["alias"]`, `.iter()`, or `.latest()`.

```python
data = query.data(cast_value="float")
print(data)                  # DataObject(lazy, ~8760 rows, range=... to ..., aliases=['flow'], ...)
print(data.total_rows)       # 8760  — no data fetched yet
print(data.time_range)       # (datetime(...), datetime(...))
data.metadata()              # works without fetching data

df = data.dataframe()        # NOW the time-series is fetched
```

This notebook walks through the key features using the **Benicia wastewater treatment plant** model.

## Setup

Connect to a running Acquirium server and load the test graph. Make sure containers are running (`make up` or `make testing-up`).

In [1]:
import time
from acquirium import Acquirium, DataObject
from acquirium.Client.query import Query
from acquirium.internals.internals_namespaces import WATR, S223, QUDT_QUANTITY_KIND, UNIT
import os

#change cwd to project root 
os.chdir("../")

acq = Acquirium(server_url="localhost", server_port=8000, use_ssl=False)
print(os.getcwd())
# Load the Benicia wastewater treatment plant model
acq.insert_graph("deployments/BENICIA/benicia-model-with-refs-thresholds.ttl")

# Wait for ingestion to complete
time.sleep(1)
status = acq.client.ingest_status()
while status["done"] < status["total"] - status["error"]:
    time.sleep(2)
    status = acq.client.ingest_status()
print(f"Ingestion complete: {status}")

/Users/umsaka/Documents/DDCPS/acquirium


INFO:acquirium.Client.client:acquirium client: external references ingested: {'ok': 28, 'skipped': 0, 'failed': 0, 'total': 28}


Ingestion complete: {'scheduled': 0, 'done': 38, 'error': 0, 'total': 38}


## 1. Basic Usage — `query.data()` is Lazy

Call `.data()` on any query that has data nodes. This returns a **lazy** `DataObject` — only metadata and per-series stats are fetched, not the actual time-series data.

In [2]:
# Find all data points in the graph
query = acq.find_all_data()
query.metadata_head()

# Get a lazy DataObject — no timeseries data is fetched yet
data = query.data(limit=10)
print(data)
print(f"\nLazy stats (no data fetched):")
print(f"  total_rows:   ~{data.total_rows}")
print(f"  time_range:   {data.time_range}")
print(f"  aliases:      {data.aliases}")
print(f"  is_empty:     {data.is_empty()}")
print(f"  materialized: {data._materialized}")

                                 Metadata First 10 Rows                                  
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ 0                                    ┃ 0_ref                                          ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ Influent_Pump-in-tss-concentration   │ Influent_Pump-in-tss-concentration_mqtt_ref    │
│ Effluent_Pump-out-ph                 │ Effluent_Pump-out-ph_mqtt_ref                  │
│ Sluice_Gate-status                   │ Sluice_Gate-status_parquet_ref                 │
│ RBC_Secondary_Sedimentation-volume   │ RBC_Secondary_Sedimentation-volume_parquet_ref │
│ Effluent_Pump-out-teq-dioxin         │ Effluent_Pump-out-teq-dioxin_parquet_ref       │
│ AS_Aeration_Basin-mlss-concentration │ AS_Aeration_Basin-mlss-concentration_mqtt_ref  │
│ Effluent_Pump-out-flow-rate          │ Effluent_Pump-out-flow-rate_mqtt_ref           │
│ Effluent_Pump-out-nh4-mgL            │ Effluent_Pump-out-nh4-mgL_mqtt_ref             │
│ Effluent_Pump-out-cl2-mgL            │ Effluent_Pump-out-cl2-mgL_mqtt_ref             │
│ Chlorination_Basin-cl2-mgL           │ Chlorination_Basin-cl2-mgL_parquet_ref         │
└──────────────────────────────────────┴────────────────────────────────────────────────┘

DataObject(lazy, ~280000 rows, range=2026-01-27T23:30:16.668982+00:00 to 2026-02-03T22:09:16.668982+00:00, aliases=['0'], entities=[])

Lazy stats (no data fetched):
  total_rows:   ~280000
  time_range:   (datetime.datetime(2026, 1, 27, 23, 30, 16, 668982, tzinfo=TzInfo(0)), datetime.datetime(2026, 2, 3, 22, 9, 16, 668982, tzinfo=TzInfo(0)))
  aliases:      ['0']
  is_empty:     False
  materialized: False


## 2. Lazy Introspection — Metadata Without Fetching Data

`metadata()`, `aliases`, `ref_info()`, `total_rows`, `time_range`, and `is_empty()` all work without triggering materialization. Per-binding stats let you inspect shape before committing to a fetch.

In [3]:
# Build a query: find all Pumps and their sensor data
pump_query = (
    acq.find_entity(_class=WATR.Pump, alias="pump")
        .find_data(alias="sensor")
)
pump_query.show_query_graph()

data = pump_query.data(limit=5, order="desc", cast_value="float")
print(data)  # shows "lazy" with approximate row count and time range

# All of these work WITHOUT fetching time-series data:
print(f"\nAliases: {data.aliases}")
print(f"Entity aliases: {data.entity_aliases}")
print(f"Total rows: ~{data.total_rows}")
print(f"Time range: {data.time_range}")

# Metadata also works without materialization
print("\nMetadata (no data fetched yet):")
print(data.metadata())
print(f"\nStill lazy? {not data._materialized}")

QUERY GRAPH

Nodes:
  0 [pump]  class=urn:nawi-water-ontology#Pump
  2 [sensor] [DATA]  class=*

Edges:
  pump --(*, hops=1)--> sensor

Data nodes:
  2 [sensor]  filters={}

Current pointer: sensor

DataObject(lazy, ~160000 rows, range=2026-01-27T23:30:16.668982+00:00 to 2026-02-03T22:09:16.668982+00:00, aliases=['sensor'], entities=['pump'])

Aliases: ['sensor']
Entity aliases: ['pump']
Total rows: ~160000
Time range: (datetime.datetime(2026, 1, 27, 23, 30, 16, 668982, tzinfo=TzInfo(0)), datetime.datetime(2026, 2, 3, 22, 9, 16, 668982, tzinfo=TzInfo(0)))

Metadata (no data fetched yet):
shape: (32, 4)
┌────────────┬───────────────────────────────┬──────────────────────────────┬──────────────────────┐
│ data_alias ┆ point_uri                     ┆ ref_uri                      ┆ entity__pump         │
│ ---        ┆ ---                           ┆ ---                          ┆ ---                  │
│ str        ┆ str                           ┆ str                          ┆ str      

## 3. Materialization on Access — `data["alias"]`

Accessing time-series by alias triggers automatic materialization. The first access fetches data; subsequent accesses use the cached result.

In [4]:
# Accessing data["alias"] triggers materialization
print(f"Before access: materialized={data._materialized}")

sensor_df = data["sensor"]
print(f"After access:  materialized={data._materialized}")
print(f"\ndata['sensor']:")
sensor_df

Before access: materialized=False
After access:  materialized=True

data['sensor']:


time,value,ref_uri
"datetime[μs, UTC]",f64,str
2026-02-03 22:05:16.668982 UTC,247.766218,"""urn:ex/Influent_Pump-in-tss-co…"
2026-02-03 22:05:16.668982 UTC,3.652556,"""urn:ex/Influent_Pump-in-flow-r…"
2026-02-03 22:05:16.668982 UTC,287.730348,"""urn:ex/Influent_Pump-in-bioche…"
2026-02-03 22:05:16.668982 UTC,6.576903,"""urn:ex/Influent_Pump-in-cyanid…"
2026-02-03 22:05:16.668982 UTC,16.846925,"""urn:ex/Effluent_Pump-out-tss-c…"
…,…,…
2026-02-03 22:09:16.668982 UTC,16.378669,"""urn:ex/Effluent_Pump-out-bioch…"
2026-02-03 22:09:16.668982 UTC,17.800925,"""urn:ex/Effluent_Pump-out-cyani…"
2026-02-03 22:09:16.668982 UTC,8.996803,"""urn:ex/Effluent_Pump-out-ph_pa…"


## 4. Lazy Grouping by Entity — `data.by("alias")`

When your query includes entity nodes, `by()` groups data by entity. **Lazy DataObjects yield lazy sub-objects** — each group only materializes when you access its data.

In [5]:
# Create a fresh lazy DataObject to demonstrate lazy grouping
data = pump_query.data(limit=5, order="desc", cast_value="float")
print(f"Parent DataObject: {data}")
print(f"Parent materialized: {data._materialized}\n")

# by() on a lazy DataObject yields lazy sub-objects
for pump_uri, group in data.by("pump"):
    print(f"Pump: {pump_uri}")
    print(f"  Sub-object: {group}")
    print(f"  Materialized: {group._materialized}")
    print(f"  Total rows: ~{group.total_rows}")

    # Accessing the data triggers materialization of just this group
    sensor_df = group.dataframe(shape="wide")
    print(f"  After .dataframe(): materialized={group._materialized}")
    print(f"  Rows: {len(sensor_df)}")
    print(sensor_df)
    print()

Parent DataObject: DataObject(lazy, ~160000 rows, range=2026-01-27T23:30:16.668982+00:00 to 2026-02-03T22:09:16.668982+00:00, aliases=['sensor'], entities=['pump'])
Parent materialized: False

Pump: urn:ex/Effluent_Pump
  Sub-object: DataObject(lazy, ~110000 rows, range=2026-01-27T23:30:16.668982+00:00 to 2026-02-03T22:09:16.668982+00:00, aliases=['sensor'], entities=['pump'])
  Materialized: False
  Total rows: ~110000
  After .dataframe(): materialized=True
  Rows: 5
shape: (5, 12)
┌────────────┬───────────┬──────────┬───────────┬───┬───────────┬───────────┬──────────┬───────────┐
│ time       ┆ sensor_9  ┆ sensor_0 ┆ sensor_3  ┆ … ┆ sensor_1  ┆ sensor_4  ┆ sensor_7 ┆ sensor_10 │
│ ---        ┆ ---       ┆ ---      ┆ ---       ┆   ┆ ---       ┆ ---       ┆ ---      ┆ ---       │
│ datetime[μ ┆ f64       ┆ f64      ┆ f64       ┆   ┆ f64       ┆ f64       ┆ f64      ┆ f64       │
│ s, UTC]    ┆           ┆          ┆           ┆   ┆           ┆           ┆          ┆           │
╞═════

## 5. Flat DataFrames — `data.dataframe()`

Call `.dataframe()` to materialize and get a standard wide or narrow DataFrame for plotting or analysis.

In [6]:
# Fresh lazy DataObject — find Chlorination Basin concentration data
cl2_query = (
    acq.find_entity(_class=WATR.ChlorinationBasin, alias="basin")
        .find_data(alias="measurement")
)

data = cl2_query.data(limit=5, order="desc", cast_value="float")

# Wide format: [time, measurement_0, measurement_1, ...]
wide_df = data.dataframe(shape="wide")
print("Wide DataFrame:")
wide_df

Wide DataFrame:


time,measurement_1,measurement_0
"datetime[μs, UTC]",f64,f64
2026-02-03 22:05:16.668982 UTC,601226.759643,72.39182
2026-02-03 22:06:16.668982 UTC,581327.27946,70.502666
2026-02-03 22:07:16.668982 UTC,610583.608297,68.760561
2026-02-03 22:08:16.668982 UTC,576623.421957,67.346247
2026-02-03 22:09:16.668982 UTC,572956.135585,68.716004


In [7]:
# Narrow format: the full enriched tall frame
narrow_df = data.dataframe(shape="narrow")
print("Narrow DataFrame:")
narrow_df

Narrow DataFrame:


data_alias,point_uri,ref_uri,entity__basin,time,value
str,str,str,str,"datetime[μs, UTC]",f64
"""measurement""","""urn:ex/Chlorination_Basin-volu…","""urn:ex/Chlorination_Basin-volu…","""urn:ex/Chlorination_Basin""",2026-02-03 22:05:16.668982 UTC,601226.759643
"""measurement""","""urn:ex/Chlorination_Basin-cl2-…","""urn:ex/Chlorination_Basin-cl2-…","""urn:ex/Chlorination_Basin""",2026-02-03 22:05:16.668982 UTC,72.39182
"""measurement""","""urn:ex/Chlorination_Basin-volu…","""urn:ex/Chlorination_Basin-volu…","""urn:ex/Chlorination_Basin""",2026-02-03 22:06:16.668982 UTC,581327.27946
"""measurement""","""urn:ex/Chlorination_Basin-cl2-…","""urn:ex/Chlorination_Basin-cl2-…","""urn:ex/Chlorination_Basin""",2026-02-03 22:06:16.668982 UTC,70.502666
"""measurement""","""urn:ex/Chlorination_Basin-volu…","""urn:ex/Chlorination_Basin-volu…","""urn:ex/Chlorination_Basin""",2026-02-03 22:07:16.668982 UTC,610583.608297
"""measurement""","""urn:ex/Chlorination_Basin-cl2-…","""urn:ex/Chlorination_Basin-cl2-…","""urn:ex/Chlorination_Basin""",2026-02-03 22:07:16.668982 UTC,68.760561
"""measurement""","""urn:ex/Chlorination_Basin-volu…","""urn:ex/Chlorination_Basin-volu…","""urn:ex/Chlorination_Basin""",2026-02-03 22:08:16.668982 UTC,576623.421957
"""measurement""","""urn:ex/Chlorination_Basin-cl2-…","""urn:ex/Chlorination_Basin-cl2-…","""urn:ex/Chlorination_Basin""",2026-02-03 22:08:16.668982 UTC,67.346247
"""measurement""","""urn:ex/Chlorination_Basin-volu…","""urn:ex/Chlorination_Basin-volu…","""urn:ex/Chlorination_Basin""",2026-02-03 22:09:16.668982 UTC,572956.135585


## 6. Iterating Individual Series — `data.iter("alias")`

When you need to process each point's timeseries individually (e.g., per-sensor anomaly detection).

In [8]:
# Iterate over each individual point's timeseries for the pump sensors
data = pump_query.data(limit=5, order="desc", cast_value="float")
for point_uri, series_df in data.iter("sensor"):
    print(f"Point: {point_uri}  |  rows: {len(series_df)}  |  mean: {series_df['value'].mean():.2f}")

Point: urn:ex/Effluent_Pump-out-bacteria-enterococcus  |  rows: 5  |  mean: 210.00
Point: urn:ex/Effluent_Pump-out-biochemical-oxygen-demand  |  rows: 5  |  mean: 16.73
Point: urn:ex/Effluent_Pump-out-cl2-mgL  |  rows: 5  |  mean: 0.36
Point: urn:ex/Effluent_Pump-out-copper  |  rows: 5  |  mean: 26.31
Point: urn:ex/Effluent_Pump-out-cyanide  |  rows: 5  |  mean: 19.00
Point: urn:ex/Effluent_Pump-out-flow-rate  |  rows: 5  |  mean: 3.37
Point: urn:ex/Effluent_Pump-out-nh4-mgL  |  rows: 5  |  mean: 16.07
Point: urn:ex/Effluent_Pump-out-ph  |  rows: 5  |  mean: 9.23
Point: urn:ex/Effluent_Pump-out-teq-dioxin  |  rows: 5  |  mean: 0.00
Point: urn:ex/Effluent_Pump-out-tss-concentration  |  rows: 5  |  mean: 16.50
Point: urn:ex/Effluent_Pump-speed-command  |  rows: 5  |  mean: 78.34
Point: urn:ex/Influent_Pump-in-biochemical-oxygen-demand  |  rows: 5  |  mean: 403.55
Point: urn:ex/Influent_Pump-in-cyanide  |  rows: 5  |  mean: 6.23
Point: urn:ex/Influent_Pump-in-flow-rate  |  rows: 5  |  mea

## 7. Metadata & Introspection (No Materialization)

All introspection methods work from binding metadata — no time-series data is fetched.

In [9]:
# Fresh lazy DataObject
data = pump_query.data(cast_value="float")

# Metadata: unique combinations of alias, point_uri, ref_uri, and entity URIs
# This does NOT trigger materialization
print("Metadata (lazy):")
print(data.metadata())
print(f"\nStill lazy: {not data._materialized}")

Metadata (lazy):
shape: (32, 4)
┌────────────┬───────────────────────────────┬──────────────────────────────┬──────────────────────┐
│ data_alias ┆ point_uri                     ┆ ref_uri                      ┆ entity__pump         │
│ ---        ┆ ---                           ┆ ---                          ┆ ---                  │
│ str        ┆ str                           ┆ str                          ┆ str                  │
╞════════════╪═══════════════════════════════╪══════════════════════════════╪══════════════════════╡
│ sensor     ┆ urn:ex/Effluent_Pump-out-cl2- ┆ urn:ex/Effluent_Pump-out-cl2 ┆ urn:ex/Effluent_Pump │
│            ┆ m…                            ┆ -m…                          ┆                      │
│ sensor     ┆ urn:ex/Influent_Pump-in-flow- ┆ urn:ex/Influent_Pump-in-flow ┆ urn:ex/Influent_Pump │
│            ┆ r…                            ┆ -r…                          ┆                      │
│ sensor     ┆ urn:ex/Effluent_Pump-out-bact ┆ urn:ex/Efflu

In [10]:
# Per-binding stats — also lazy
print(f"Total rows across all series: ~{data.total_rows}")
print(f"Time range: {data.time_range}")
print(f"\nPer-binding details:")
for b in data.bindings:
    print(f"  {b.alias} | {b.ref_uri} | ~{b.row_count} rows | {b.earliest} to {b.latest}")

print(f"\nRef info for 'sensor':")
for idx, ref_uri in data.ref_info("sensor"):
    print(f"  [{idx}] {ref_uri}")

print(f"\nStill lazy: {not data._materialized}")

Total rows across all series: ~160000
Time range: (datetime.datetime(2026, 1, 27, 23, 30, 16, 668982, tzinfo=TzInfo(0)), datetime.datetime(2026, 2, 3, 22, 9, 16, 668982, tzinfo=TzInfo(0)))

Per-binding details:
  sensor | urn:ex/Influent_Pump-in-tss-concentration_mqtt_ref | ~0 rows | None to None
  sensor | urn:ex/Influent_Pump-in-tss-concentration_parquet_ref | ~10000 rows | 2026-01-27 23:30:16.668982+00:00 to 2026-02-03 22:09:16.668982+00:00
  sensor | urn:ex/Influent_Pump-in-flow-rate_mqtt_ref | ~0 rows | None to None
  sensor | urn:ex/Influent_Pump-in-flow-rate_parquet_ref | ~10000 rows | 2026-01-27 23:30:16.668982+00:00 to 2026-02-03 22:09:16.668982+00:00
  sensor | urn:ex/Influent_Pump-in-biochemical-oxygen-demand_parquet_ref | ~10000 rows | 2026-01-27 23:30:16.668982+00:00 to 2026-02-03 22:09:16.668982+00:00
  sensor | urn:ex/Influent_Pump-in-biochemical-oxygen-demand_mqtt_ref | ~0 rows | None to None
  sensor | urn:ex/Influent_Pump-in-cyanide_mqtt_ref | ~0 rows | None to None
 

In [11]:
# Latest value for an alias (triggers materialization)
print("Latest value:")
data.latest("sensor")

Latest value:


time,value
"datetime[μs, UTC]",f64
2026-02-03 22:09:16.668982 UTC,265.114607


## 8. Multi-Level Query — Sedimentation Tanks and Downstream Equipment

A more complex query with multiple entity levels and data nodes, showing how grouping and alias access compose together.

In [13]:
# Multi-level: find Sedimentation Tanks, their downstream equipment, and all data
multi_query = (
    acq.find_entity(_class=WATR.SedimentationTank, alias="tank")
       .find_related(_class="static mixer",alias="downstream", direction="downstream")
       .find_all_data(alias="readings")
)
multi_query.show_query_graph()

multi_data = multi_query.data(limit=3, order="desc", cast_value="float")
print(multi_data)
print("\nEntity aliases:", multi_data.entity_aliases)

QUERY GRAPH

Nodes:
  0 [tank]  class=urn:nawi-water-ontology#SedimentationTank
  2 [downstream]  class=urn:nawi-water-ontology#StaticMixer
  4 [readings] [DATA]  class=*
  6 [readings_1] [DATA]  class=*

Edges:
  tank --(direction=downstream, hops=3)--> downstream
  tank --(*, hops=1)--> readings
  downstream --(*, hops=1)--> readings_1

Data nodes:
  4 [readings]  filters={}
  6 [readings_1]  filters={}

Current pointer: readings_1

DataObject(0 rows, aliases=[], entities=[])

Entity aliases: []


In [14]:
# Group by tank and summarize readings
if "tank" in multi_data.entity_aliases:
    for tank_uri, group in multi_data.by("tank"):
        for alias in group.aliases:
            readings = group[alias]
            if not readings.is_empty():
                print(f"Tank: {tank_uri}  |  alias: {alias}")
                print(f"  Latest value: {readings.sort('time', descending=True)['value'][0]}")
                print(f"  Mean: {readings['value'].mean():.2f}")
                print()

## 9. App Pattern — Effluent Monitoring (Before vs After)

The `DataObject` API makes app logic much more readable and robust.

In [15]:
# --- BEFORE: fragile positional indexing ---
# df = query.latest_data(cast_value='float')
# if df[0,1] > 75:   # What is column 1? What if column order changes?
#     print("Alert!")

# --- AFTER: alias-driven with lazy DataObject ---
# Find the Effluent Pump and its outgoing sensor data
effluent_query = (
    acq.find_entity(_class=WATR.Pump, alias="pump")
        .find_data(alias="effluent_sensor")
)

data = effluent_query.data(limit=1, order="desc", cast_value="float")

# Inspect lazily first — no data fetched
print(f"Found {len(data.bindings)} sensor bindings across {len(data.aliases)} aliases")
print(f"Approximate rows: ~{data.total_rows}\n")

# Now materialize per pump group
for pump_uri, group in data.by("pump"):
    sensor = group["effluent_sensor"]
    if sensor.is_empty():
        print(f"{pump_uri}: No data")
    else:
        val = sensor["value"][0]
        ts = sensor["time"][0]
        print(f"{pump_uri}: value={val:.2f} at {ts}")

Found 32 sensor bindings across 1 aliases
Approximate rows: ~160000

urn:ex/Effluent_Pump: value=16.39 at 2026-02-03 22:09:16.668982+00:00
urn:ex/Influent_Pump: value=265.11 at 2026-02-03 22:09:16.668982+00:00


## API Reference

### Lazy (no data fetched)

| Method / Property | Returns | Description |
|---|---|---|
| `query.data(start, end, limit, order, cast_value)` | `DataObject` | Construct a lazy DataObject (SPARQL + stats only) |
| `data.aliases` | `list[str]` | Available data aliases |
| `data.entity_aliases` | `list[str]` | Available entity aliases |
| `data.metadata()` | `pl.DataFrame` | Unique `(data_alias, point_uri, ref_uri, entity__*)` |
| `data.ref_info("alias")` | `list[(int, str)]` | Indexed ref URIs |
| `data.is_empty()` | `bool` | Whether any data exists |
| `data.total_rows` | `int` | Approximate total row count across all series |
| `data.time_range` | `tuple[datetime, datetime]` | Overall `(earliest, latest)` across all series |
| `data.bindings` | `list[BindingInfo]` | Per-series metadata with stats |
| `data.by("entity_alias")` | `Iterator[(str, DataObject)]` | Group by entity — yields lazy sub-objects |

### Materializing (triggers data fetch on first call)

| Method / Property | Returns | Description |
|---|---|---|
| `data["alias"]` | `pl.DataFrame` | `[time, value]` for single-ref; `[time, value, ref_uri]` for multi-ref |
| `data.dataframe(shape="wide")` | `pl.DataFrame` | Pivoted `[time, alias_1, alias_2, ...]` |
| `data.dataframe(shape="narrow")` | `pl.DataFrame` | Full enriched tall frame |
| `data.iter("alias")` | `Iterator[(str, pl.DataFrame)]` | Per-point `(point_uri, [time, value])` |
| `data.latest("alias")` | `pl.DataFrame` | Most recent `[time, value]` |